In [1]:
!pip install -q datasets sentence-transformers pandas numpy torch tqdm

In [2]:
# %% [markdown]
# # Lightweight Embedding Models Evaluation on LitSearchRetrieval
# This notebook evaluates fast, low-parameter embedding models on the `mteb/LitSearchRetrieval` dataset. 
# It calculates Recall@K and MRR using optimized PyTorch matrix operations for lightning-fast retrieval.

# %% [code]
# 1. INSTALL DEPENDENCIES
# Run this cell if you haven't installed these libraries in your environment yet
# !pip install -q datasets sentence-transformers pandas numpy torch tqdm

# %% [code]
# 2. IMPORTS & SETUP
import os
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from tqdm.auto import tqdm

# Set device to GPU if available for faster matrix operations
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {DEVICE}")

🚀 Using device: cuda


In [3]:
# %% [code]
# 3. LOAD & PREPARE DATASET
print("⏳ Loading mteb/LitSearchRetrieval dataset...")

# MTEB datasets generally use the "default" configuration for BEIR format
corpus_dataset = load_dataset("mteb/LitSearchRetrieval", "corpus")
queries_dataset = load_dataset("mteb/LitSearchRetrieval", "queries")
qrels_dataset = load_dataset("mteb/LitSearchRetrieval", "qrels") # Usually test split for qrels

⏳ Loading mteb/LitSearchRetrieval dataset...


README.md: 0.00B [00:00, ?B/s]

corpus/test-00000-of-00001.parquet:   0%|          | 0.00/34.0M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/64183 [00:00<?, ? examples/s]

queries/test-00000-of-00001.parquet:   0%|          | 0.00/47.4k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/597 [00:00<?, ? examples/s]

qrels/test-00000-of-00001.parquet:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/639 [00:00<?, ? examples/s]

In [4]:
# Safely extract the first available split and convert to pandas
corpus_df = corpus_dataset[list(corpus_dataset.keys())[0]].to_pandas()
queries_df = queries_dataset[list(queries_dataset.keys())[0]].to_pandas()
qrels_df = qrels_dataset[list(qrels_dataset.keys())[0]].to_pandas()
# Normalize ID columns based on the schema you provided (handling potential '_id' vs 'id' discrepancies)
corpus_id_col = '_id' if '_id' in corpus_df.columns else 'id'
query_id_col = '_id' if '_id' in queries_df.columns else 'id'

# Create lookup structures
print(f"📊 Dataset Size -> Corpus: {len(corpus_df)}, Queries: {len(queries_df)}, Qrels: {len(qrels_df)}")

📊 Dataset Size -> Corpus: 64183, Queries: 597, Qrels: 639


In [5]:
# Map corpus IDs to their row index in the DataFrame for fast indexing later
corpus_id_to_idx = {cid: idx for idx, cid in enumerate(corpus_df[corpus_id_col])}

# Build a dictionary mapping query-id to a set of relevant corpus-ids (where score > 0)
ground_truth = defaultdict(set)
for _, row in qrels_df.iterrows():
    q_id = row['query-id']
    c_id = row['corpus-id']
    score = row['score']
    if score > 0:  # We only consider positive relations as ground truth
        ground_truth[q_id].add(c_id)

# Filter out queries that don't have any ground truth (to ensure fair evaluation)
valid_query_ids = set(ground_truth.keys())
queries_df = queries_df[queries_df[query_id_col].isin(valid_query_ids)].reset_index(drop=True)
print(f"🎯 Evaluating on {len(queries_df)} valid queries with known ground truth.")

# Combine Title and Text for the corpus to give the model maximum context
corpus_texts = (corpus_df['title'].fillna('') + ' ' + corpus_df['text'].fillna('')).tolist()
query_texts = queries_df['text'].tolist()
query_ids = queries_df[query_id_col].tolist()

🎯 Evaluating on 597 valid queries with known ground truth.


In [6]:
MODELS_TO_TEST = [
    "google-bert/bert-base-uncased",
    "sentence-transformers/all-MiniLM-L6-v2", # 22M params
    "BAAI/bge-small-en-v1.5",                 # 33M params
    "thenlper/gte-small"
]
for model in MODELS_TO_TEST :
    test_model = SentenceTransformer(model, device=DEVICE)

No sentence-transformers model found with name google-bert/bert-base-uncased. Creating a new one with mean pooling.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
# %% [code]
# 4. EVALUATION LOGIC
def evaluate_model(model_name: str, k_values: list = [1, 5, 10]):
    print(f"\n" + "="*50)
    print(f"🧠 Loading Model: {model_name}")
    print("="*50)
    
    # Load model
    model = SentenceTransformer(model_name, device=DEVICE)
    
    # Encode Corpus (normalize_embeddings=True ensures we can use fast Dot Product instead of full Cosine)
    print("   -> Encoding Corpus...")
    corpus_embeddings = model.encode(
        corpus_texts, batch_size=128, show_progress_bar=True, 
        convert_to_tensor=True, normalize_embeddings=True
    )
    
    # Encode Queries
    print("   -> Encoding Queries...")
    query_embeddings = model.encode(
        query_texts, batch_size=128, show_progress_bar=False, 
        convert_to_tensor=True, normalize_embeddings=True
    )
    
    # Compute Similarities (Since embeddings are normalized, Dot Product == Cosine Similarity)
    print("   -> Computing Cosine Similarities...")
    similarities = torch.mm(query_embeddings, corpus_embeddings.T) # shape: (num_queries, num_corpus)
    
    # Get top-K indices
    max_k = max(k_values)
    top_k_scores, top_k_indices = torch.topk(similarities, k=max_k, dim=1)
    top_k_indices = top_k_indices.cpu().numpy()
    
    # Calculate Metrics
    print("   -> Calculating Metrics...")
    metrics = {f"Recall@{k}": [] for k in k_values}
    metrics.update({f"MRR@{k}": [] for k in k_values})
    
    for i, q_id in enumerate(query_ids):
        relevant_docs = ground_truth[q_id]
        if not relevant_docs:
            continue
            
        retrieved_indices = top_k_indices[i]
        
        # Map indices back to actual corpus IDs
        retrieved_doc_ids = [corpus_df.iloc[idx][corpus_id_col] for idx in retrieved_indices]
        
        for k in k_values:
            k_retrieved = retrieved_doc_ids[:k]
            
            # 1. Recall@K: Did we find at least one relevant document in the top K? (or fraction of total relevant)
            # Standard MTEB Recall calculates the proportion of relevant docs found.
            hits = len(set(k_retrieved).intersection(relevant_docs))
            recall = hits / len(relevant_docs)
            metrics[f"Recall@{k}"].append(recall)
            
            # 2. MRR@K: Multi Reciprocal Rank
            mrr = 0.0
            for rank, doc_id in enumerate(k_retrieved):
                if doc_id in relevant_docs:
                    mrr = 1.0 / (rank + 1)
                    break # Only care about the first relevant hit for MRR
            metrics[f"MRR@{k}"].append(mrr)
            
    # Aggregate results
    aggregated_results = {metric: np.mean(scores) for metric, scores in metrics.items()}
    return aggregated_results

# %% [code]
# 5. RUN EXPERIMENTS
MODELS_TO_TEST = [
    "google-bert/bert-base-uncased",
    "sentence-transformers/all-MiniLM-L6-v2", # 22M params
    "BAAI/bge-small-en-v1.5",                 # 33M params
    "thenlper/gte-small"
]

all_results = []

for model in MODELS_TO_TEST:
    results = evaluate_model(model, k_values=[1, 5, 10])
    results["Model"] = model
    all_results.append(results)

No sentence-transformers model found with name google-bert/bert-base-uncased. Creating a new one with mean pooling.



🧠 Loading Model: google-bert/bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   -> Encoding Corpus...


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   -> Encoding Corpus...


Batches:   0%|          | 0/502 [00:00<?, ?it/s]

   -> Encoding Queries...
   -> Computing Cosine Similarities...
   -> Calculating Metrics...


In [8]:
# %% [code]
# 6. FINAL RESULTS SUMMARY
results_df = pd.DataFrame(all_results)

# Reorder columns for readability
cols = ["Model", "MRR@10", "Recall@10", "MRR@5", "Recall@5", "MRR@1", "Recall@1"]
results_df = results_df[[c for c in cols if c in results_df.columns]]

print("\n" + "🌟" * 25)
print(" FINAL BENCHMARK RESULTS")
print("🌟" * 25)
# Display formatted markdown table in outputs
display(results_df.round(4))


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
 FINAL BENCHMARK RESULTS
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟


,Model,MRR@10,Recall@10,MRR@5,Recall@5,MRR@1,Recall@1
0,google-bert/bert-base-uncased,0.0115,0.0268,0.0092,0.0126,0.0067,0.0067
1,sentence-transformers/all-MiniLM-L6-v2,0.3108,0.5045,0.2998,0.4174,0.2295,0.2217
2,BAAI/bge-small-en-v1.5,0.3966,0.5705,0.3855,0.4942,0.3049,0.2951
3,thenlper/gte-small,0.3873,0.5570,0.3783,0.4875,0.3099,0.2988
